In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [ ]:
from hotelling.spatial.census import build_grid_polygons

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Pop grid was saved with point geometry (midpoints). Convert to 100m square polygons.
grid = build_grid_polygons(grid)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells as square polygons")


In [ ]:
from hotelling.spatial.city_data import process_ihk_data

grid = process_ihk_data(grid, PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv')
print(f"Grid with IHK employment: empl column added, total empl={grid['empl'].sum():.0f}")


In [ ]:
from hotelling.spatial.city_data import process_gebaeude_stadtstruktur

gebaeude_stadtstruktur = process_gebaeude_stadtstruktur(
    gebaeude_path=PATH_RAW / 'gebaeude.gpkg',
    stadtstruktur_path=PATH_RAW / 'stadtstruktur.gpkg',
    ihk_path=PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv',
)
print(f"gebaeude_stadtstruktur: {len(gebaeude_stadtstruktur)} buildings")
print(f"  approx_empl total: {gebaeude_stadtstruktur['approx_empl'].sum():.0f}")
print(f"  buildings with empl>0: {(gebaeude_stadtstruktur['approx_empl'] > 0).sum()}")


In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
import matplotlib.pyplot as plt
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red', markersize=10)
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red')
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [ ]:
# IHK→building distance matching is performed inside process_gebaeude_stadtstruktur().
# Inspect gebaeude_stadtstruktur['empl'] and ['approx_empl'] for capped employment totals.
gebaeude_stadtstruktur[['empl', 'approx_empl', 'employee_hard_cap']].describe()


In [ ]:
import matplotlib.pyplot as plt

gebaeude_centroid = gebaeude_stadtstruktur.copy()
gebaeude_centroid['geometry'] = gebaeude_centroid.geometry.centroid

# Handle NaN values and normalization safely
max_empl = gebaeude_centroid['approx_empl'].replace([np.inf, -np.inf], np.nan).max()
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl'] / max_empl if max_empl > 0 else 0
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl_norm'].fillna(0)

if GENERATE_PLOTS:
    # Plot buildings colored by approx_empl with alpha dependent on approx_empl_norm
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Filter to non-zero employment buildings
    to_plot = gebaeude_centroid[gebaeude_centroid['approx_empl_norm'] != 0].copy()

    # Extract coordinates
    x = to_plot.geometry.x.values
    y = to_plot.geometry.y.values
    c = to_plot['approx_empl_norm'].values

    # Make alpha dependent on approx_empl_norm (scale to [0.1, 0.9] for visibility)
    v_min = np.vectorize(lambda x, y: min(x, y))
    alphas = v_min(0.05 + 0.1 * np.exp(c), 1)

    # Plot with variable alpha
    cmap = plt.get_cmap('viridis')
    scatter = ax.scatter(x, y, c=c, cmap=cmap, s=5, alpha=alphas, vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label='Normalized Employment')

    berlin.to_crs(gebaeude_stadtstruktur.crs).plot(ax=ax, color='none', edgecolor='black', linewidth=1)

    plt.title('Buildings colored and glowing by approximate employment')
    plt.show()


In [ ]:
from hotelling.spatial.city_data import run_prime_location_clustering

prime_location_clusters = run_prime_location_clustering(
    gebaeude_stadtstruktur,
    k_percentile=99.5,
    min_empl=10,
    radius_m=500,
)
print(f"Prime-location clusters: {len(prime_location_clusters)}")
prime_location_clusters


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fix, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Add OSM basemap
    result_web = prime_location_clusters.to_crs(berlin.crs)
    # Plot clusters and boundary
    result_web.plot(ax=ax, column='cluster_id', markersize=5, legend=True, alpha=1)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=result_web.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)

    plt.title('Detected employment clusters with OSM background')
    plt.show()


In [ ]:
# Outputs are saved automatically by the package functions:
# data/processed/gebaeude_stadtstruktur.parquet  ← process_gebaeude_stadtstruktur()
# data/processed/prime_location_clusters.parquet ← run_prime_location_clustering()
print("Outputs saved by package functions.")


In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_02_city_data.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")